In [1]:
import time
import pandas as pd
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# --- 1. CONFIGURACIÓN DE METADATOS Y MONGODB (Código Profe) ---
NOMBRE_GRUPO = "G1_Ecommerce"
CATEGORIA_ACTUAL = "Travel" 

try:
    # 'database' es el nombre del servicio en tu docker-compose
    client = MongoClient('database', 27017) 
    db = client['proyecto_bigdata']
    coleccion = db['datos_libros']
    print("CONEXIÓN EXITOSA A MONGODB: Listo para insertar datos.")
except Exception as e:
    print(" ERROR DE CONEXIÓN A MONGODB:", e)

# --- 2. CONFIGURACIÓN DEL NAVEGADOR (Tu código) ---
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

URL_INICIAL = "https://books.toscrape.com/"
paginas_objetivo = 2  # Puedes subirlo a 3 si quieres más datos

try:
    print(f" Conectando a: {URL_INICIAL}")
    driver.get(URL_INICIAL)
    time.sleep(3)
    
    for pagina in range(paginas_objetivo):
        print(f"\n--- Procesando Página {pagina + 1} ---")
        
        # Esperar a que carguen los libros
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "article.product_pod"))
        )
        
        productos = driver.find_elements(By.CSS_SELECTOR, "article.product_pod")
        
        for producto in productos:
            try:
                # Extraer datos de la web
                titulo = producto.find_element(By.CSS_SELECTOR, "h3 a").get_attribute("title")
                precio_raw = producto.find_element(By.CSS_SELECTOR, "p.price_color").text
                
                # --- 3. ENRIQUECIMIENTO Y ENVÍO A MONGODB (Unión de ambos) ---
                registro = {
                    "item": titulo,
                    # Limpiamos el precio para que sea un número
                    "valor": float(precio_raw.replace('£', '').replace('$', '').strip()),
                    "categoria": CATEGORIA_ACTUAL,
                    "grupo": NOMBRE_GRUPO,
                    "pagina_origen": pagina + 1,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S")
                }
                
                # Insertar en la base de datos
                coleccion.insert_one(registro)
                print(f" Guardado: {titulo} - {precio_raw}")
                
            except Exception as e:
                print(f"Error procesando un producto: {e}")
        
        # Lógica de paginación
        try:
            boton_siguiente = driver.find_element(By.CSS_SELECTOR, "li.next a")
            boton_siguiente.click()
            time.sleep(2)
        except:
            print(" No hay más páginas disponibles.")
            break

    print("\n PROCESO FINALIZADO: Todos los datos están en MongoDB.")

except Exception as e:
    print(f" Error general: {e}")

finally:
    driver.quit()
    print(" Navegador cerrado.")

CONEXIÓN EXITOSA A MONGODB: Listo para insertar datos.
 Conectando a: https://books.toscrape.com/

--- Procesando Página 1 ---
 Guardado: A Light in the Attic - £51.77
 Guardado: Tipping the Velvet - £53.74
 Guardado: Soumission - £50.10
 Guardado: Sharp Objects - £47.82
 Guardado: Sapiens: A Brief History of Humankind - £54.23
 Guardado: The Requiem Red - £22.65
 Guardado: The Dirty Little Secrets of Getting Your Dream Job - £33.34
 Guardado: The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull - £17.93
 Guardado: The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics - £22.60
 Guardado: The Black Maria - £52.15
 Guardado: Starving Hearts (Triangular Trade Trilogy, #1) - £13.99
 Guardado: Shakespeare's Sonnets - £20.66
 Guardado: Set Me Free - £17.46
 Guardado: Scott Pilgrim's Precious Little Life (Scott Pilgrim #1) - £52.29
 Guardado: Rip it Up and Start Again - £35.02
 Guardado: Our Band Could Be Your Life: S

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

# 1. Configuración de la Sesión (Versión 3.0.1 sugerida por Cami)
spark = SparkSession.builder \
    .appName("Analisis_Categorias_Alexandra") \
    .config("spark.mongodb.input.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .getOrCreate()

try:
    # 2. Carga de datos usando el formato de la versión 3.0.1
    df = spark.read.format("com.mongodb.spark.sql.DefaultSource").load()

    # 3. Limpieza de datos (Quitar nulos y valores en 0)
    df_limpio = df.filter((col("item").isNotNull()) & (col("valor") > 0))

    # 4. Reporte Final en Tabla Ordenada
    reporte_tabla = df_limpio.groupBy("categoria").agg(
        F.count("item").alias("Cant"),
        F.round(F.avg("valor"), 2).alias("Promedio"),
        F.min("valor").alias("Min"),
        F.max("valor").alias("Max")
    ).orderBy(F.desc("Promedio"))

    # 5. Mostrar Resultados
    print("\n" + "="*60)
    print("📋 REPORTE DE MERCADO - BIG DATA")
    print("="*60)
    
    reporte_tabla.show(truncate=False)
    
    print("="*60)
    print(f"Registros procesados con éxito: {df_limpio.count()}")

except Exception as e:
    print(f"❌ Error al procesar: {e}")


📋 REPORTE DE MERCADO - BIG DATA
+---------+----+--------+-----+-----+
|categoria|Cant|Promedio|Min  |Max  |
+---------+----+--------+-----+-----+
|Travel   |40  |34.96   |12.84|57.25|
+---------+----+--------+-----+-----+

Registros procesados con éxito: 40
